# Phase 6: Stage-2 ML Modelling

Train a classifier per risk factor (target = Low/Med/High code). Features = rider profile minus the factor's own defining variable(s). Models: Logistic Regression, Decision Tree, Random Forest, XGBoost. Stratified k-fold CV.

## Step 0 — Setup

In [1]:
import pandas as pd
import numpy as np
import warnings
import joblib
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

CWD = Path.cwd()
ROOT = CWD if (CWD / 'data').exists() else CWD.parent
df     = pd.read_csv(ROOT / 'data' / 'processed' / 'model_ready.csv')
risk   = pd.read_csv(ROOT / 'data' / 'processed' / 'risk_profile.csv')
MODELS = ROOT / 'outputs' / 'models'
TABLES = ROOT / 'outputs' / 'tables'
MODELS.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

print('model_ready :', df.shape)
print('risk_profile:', risk.shape)

model_ready : (182, 91)
risk_profile: (182, 14)


## Step 1 — Build the analysis frame, features, and per-target exclusions

In [2]:
targets = ['force_risk_code', 'repetition_risk_code', 'duration_risk_code',
           'vibration_risk_code', 'contact_stress_risk_code', 'posture_risk_code']
data = df.copy()
for t in targets:
    data[t] = risk[t].values

feature_pool = ['workload_score', 'fatigue_score',
                'age_ord', 'income_ord', 'education_ord', 'job_duration_ord',
                'work_hours_num', 'work_days_num', 'deliveries_num', 'rest_break_num',
                'vehicle_rank', 'carrying_contact_rank',
                'force_exertion', 'vibration_index']

# Drop each factor's defining inputs so the model cannot trivially leak the label.
# For Posture, the severity-rank match used pain (Nordic), fatigue_score, and
# work_hours_num -- excluded so the survey-side inputs that drove the matching
# cannot trivially predict the resulting Posture label.
exclude_per_target = {
    'force_risk_code':          ['force_exertion'],
    'repetition_risk_code':     ['deliveries_num', 'work_hours_num'],
    'duration_risk_code':       ['work_hours_num'],
    'vibration_risk_code':      ['vibration_index', 'vehicle_rank', 'work_hours_num'],
    'contact_stress_risk_code': ['carrying_contact_rank', 'work_hours_num'],
    'posture_risk_code':        ['fatigue_score', 'work_hours_num'],
}

for t, drops in exclude_per_target.items():
    feats = [f for f in feature_pool if f not in drops]
    print(f'{t:28s}  drop {drops}  -> {len(feats)} features')

force_risk_code               drop ['force_exertion']  -> 13 features
repetition_risk_code          drop ['deliveries_num', 'work_hours_num']  -> 12 features
duration_risk_code            drop ['work_hours_num']  -> 13 features
vibration_risk_code           drop ['vibration_index', 'vehicle_rank', 'work_hours_num']  -> 11 features
contact_stress_risk_code      drop ['carrying_contact_rank', 'work_hours_num']  -> 12 features
posture_risk_code             drop ['fatigue_score', 'work_hours_num']  -> 12 features


## Step 2 — Define models and CV strategy

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix

RNG = 42
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG)

def fresh_models():
    # class_weight='balanced' lets the model respect the L/M/H imbalance
    return {
        'LogReg':       LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RNG),
        'DecisionTree': DecisionTreeClassifier(class_weight='balanced', random_state=RNG),
        'RandomForest': RandomForestClassifier(class_weight='balanced', n_estimators=200, random_state=RNG),
        'XGBoost':      XGBClassifier(eval_metric='mlogloss', random_state=RNG, n_estimators=200, verbosity=0),
    }

list(fresh_models().keys())

['LogReg', 'DecisionTree', 'RandomForest', 'XGBoost']

## Step 3 — 5-fold CV across all 5 factors x 4 models

In [4]:
from sklearn.preprocessing import LabelEncoder

scoring = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro']

cv_rows = []
for target, drops in exclude_per_target.items():
    feats = [f for f in feature_pool if f not in drops]
    X = data[feats]
    # 0-indexed labels (XGBoost requires this; harmless for the others)
    y = LabelEncoder().fit_transform(data[target])
    for name, model in fresh_models().items():
        s = cross_validate(model, X, y, cv=cv, scoring=scoring,
                           return_train_score=True, n_jobs=-1)
        cv_rows.append({
            'target':              target.replace('_risk_code', ''),
            'model':               name,
            'cv_accuracy':         round(s['test_accuracy'].mean(),  3),
            'cv_accuracy_std':     round(s['test_accuracy'].std(),   3),
            'cv_f1_macro':         round(s['test_f1_macro'].mean(),  3),
            'cv_precision_macro':  round(s['test_precision_macro'].mean(), 3),
            'cv_recall_macro':     round(s['test_recall_macro'].mean(),    3),
            'train_accuracy':      round(s['train_accuracy'].mean(), 3),
            'overfit_gap':         round(s['train_accuracy'].mean() - s['test_accuracy'].mean(), 3),
        })

cv_df = pd.DataFrame(cv_rows)
cv_df.to_csv(TABLES / 'cv_results.csv', index=False)
cv_df

,target,model,cv_accuracy,cv_accuracy_std,cv_f1_macro,cv_precision_macro,cv_recall_macro,train_accuracy,overfit_gap
0,force,LogReg,0.522,0.056,0.474,0.477,0.506,0.662,0.140
1,force,DecisionTree,0.489,0.065,0.437,0.448,0.447,1.000,0.511
2,force,RandomForest,0.495,0.073,0.402,0.433,0.405,1.000,0.505
3,force,XGBoost,0.527,0.092,0.465,0.478,0.469,1.000,0.473
4,repetition,LogReg,0.510,0.106,0.484,0.478,0.543,0.626,0.116
5,repetition,DecisionTree,0.681,0.059,0.604,0.607,0.625,1.000,0.319
6,repetition,RandomForest,0.692,0.058,0.549,0.598,0.552,1.000,0.308
7,repetition,XGBoost,0.697,0.068,0.646,0.707,0.643,1.000,0.303
8,duration,LogReg,0.856,0.055,0.853,0.865,0.862,0.924,0.068
9,duration,DecisionTree,0.994,0.011,0.992,0.994,0.990,1.000,0.006


## Step 4 — Pick best model per factor and save pickled fits

In [5]:
best_rows = []
for target, drops in exclude_per_target.items():
    short = target.replace('_risk_code', '')
    sub = cv_df[cv_df['target'] == short].sort_values('cv_f1_macro', ascending=False)
    best_name = sub.iloc[0]['model']

    feats = [f for f in feature_pool if f not in drops]
    X = data[feats]
    y = LabelEncoder().fit_transform(data[target])

    # Refit on the full sample and save
    model = fresh_models()[best_name]
    model.fit(X, y)
    out_path = MODELS / f'best_{short}.pkl'
    joblib.dump({'model': model, 'features': feats, 'classes': sorted(set(data[target]))}, out_path)

    best_rows.append({
        'target':        short,
        'best_model':    best_name,
        'cv_accuracy':   sub.iloc[0]['cv_accuracy'],
        'cv_f1_macro':   sub.iloc[0]['cv_f1_macro'],
        'overfit_gap':   sub.iloc[0]['overfit_gap'],
        'saved_to':      str(out_path.relative_to(ROOT)),
    })

best_df = pd.DataFrame(best_rows)
best_df.to_csv(TABLES / 'best_models.csv', index=False)
best_df

,target,best_model,cv_accuracy,cv_f1_macro,overfit_gap,saved_to
0,force,LogReg,0.522,0.474,0.140,outputs\models\best_force.pkl
1,repetition,XGBoost,0.697,0.646,0.303,outputs\models\best_repetition.pkl
2,duration,XGBoost,1.000,1.000,0.000,outputs\models\best_duration.pkl
3,vibration,XGBoost,0.516,0.510,0.484,outputs\models\best_vibration.pkl
4,contact_stress,RandomForest,0.588,0.577,0.412,outputs\models\best_contact_stress.pkl
5,posture,RandomForest,0.891,0.765,0.109,outputs\models\best_posture.pkl


## Step 5 — Confusion matrices and classification reports (best model per factor)

In [6]:
cm_rows = []
report_rows = []
for _, r in best_df.iterrows():
    target_code = r['target'] + '_risk_code'
    feats = [f for f in feature_pool if f not in exclude_per_target[target_code]]
    X = data[feats]
    raw_y = data[target_code]
    y = LabelEncoder().fit_transform(raw_y)
    present_codes = sorted(set(raw_y))
    label_names = [['Low','Medium','High'][int(c)] for c in present_codes]

    model = fresh_models()[r['best_model']]
    y_pred = cross_val_predict(model, X, y, cv=cv, n_jobs=-1)

    cm = confusion_matrix(y, y_pred, labels=list(range(len(present_codes))))
    print(f"\n=== {r['target']} (best = {r['best_model']}) ===")
    print('Confusion matrix (rows = true, cols = pred):', ' / '.join(label_names))
    print(pd.DataFrame(cm, index=label_names, columns=label_names))

    rep = classification_report(y, y_pred, target_names=label_names,
                                output_dict=True, zero_division=0)
    for level in label_names + ['macro avg', 'weighted avg']:
        d = rep[level]
        report_rows.append({
            'target':    r['target'],
            'model':     r['best_model'],
            'class':     level,
            'precision': round(d['precision'], 3),
            'recall':    round(d['recall'],    3),
            'f1':        round(d['f1-score'],  3),
            'support':   int(d['support']),
        })

    for i, true_lbl in enumerate(label_names):
        for j, pred_lbl in enumerate(label_names):
            cm_rows.append({
                'target':    r['target'],
                'model':     r['best_model'],
                'true':      true_lbl,
                'predicted': pred_lbl,
                'count':     int(cm[i, j]),
            })

pd.DataFrame(cm_rows).to_csv(TABLES / 'confusion_matrices.csv', index=False)
report_df = pd.DataFrame(report_rows)
report_df.to_csv(TABLES / 'classification_reports.csv', index=False)
report_df


=== force (best = LogReg) ===
Confusion matrix (rows = true, cols = pred): Low / Medium / High
        Low  Medium  High
Low      59      20    11
Medium   21      15    21
High      5       9    21

=== repetition (best = XGBoost) ===
Confusion matrix (rows = true, cols = pred): Low / Medium / High
        Low  Medium  High
Low      54      22     3
Medium   17      64     3
High      4       6     9



=== duration (best = XGBoost) ===
Confusion matrix (rows = true, cols = pred): Low / Medium / High
        Low  Medium  High
Low      37       0     0
Medium    0      56     0
High      0       0    89

=== vibration (best = XGBoost) ===
Confusion matrix (rows = true, cols = pred): Low / Medium / High
        Low  Medium  High
Low      40      22     5
Medium   20      32    16
High      9      16    22



=== contact_stress (best = RandomForest) ===
Confusion matrix (rows = true, cols = pred): Low / Medium / High
        Low  Medium  High
Low      39      16    13
Medium   15      32    11
High      7      13    36



=== posture (best = RandomForest) ===
Confusion matrix (rows = true, cols = pred): Medium / High
        Medium  High
Medium      14    15
High         5   148


,target,model,class,precision,recall,f1,support
0,force,LogReg,Low,0.694,0.656,0.674,90
1,force,LogReg,Medium,0.341,0.263,0.297,57
2,force,LogReg,High,0.396,0.600,0.477,35
3,force,LogReg,macro avg,0.477,0.506,0.483,182
4,force,LogReg,weighted avg,0.526,0.522,0.518,182
5,repetition,XGBoost,Low,0.720,0.684,0.701,79
6,repetition,XGBoost,Medium,0.696,0.762,0.727,84
7,repetition,XGBoost,High,0.600,0.474,0.529,19
8,repetition,XGBoost,macro avg,0.672,0.640,0.653,182
9,repetition,XGBoost,weighted avg,0.696,0.698,0.695,182


## Step 6 — Phase 6 summary

In [7]:
# One row per factor: best model + headline metrics + overfit flag
summary = best_df.copy()
summary['overfit_flag'] = summary['overfit_gap'] > 0.15
summary.to_csv(TABLES / 'phase6_summary.csv', index=False)

print('Models saved to outputs/models/:')
for p in sorted(MODELS.glob('*.pkl')):
    print(' ', p.name)
print()
summary

Models saved to outputs/models/:
  best_contact_stress.pkl
  best_duration.pkl
  best_force.pkl
  best_posture.pkl
  best_repetition.pkl
  best_vibration.pkl



,target,best_model,cv_accuracy,cv_f1_macro,overfit_gap,saved_to,overfit_flag
0,force,LogReg,0.522,0.474,0.140,outputs\models\best_force.pkl,False
1,repetition,XGBoost,0.697,0.646,0.303,outputs\models\best_repetition.pkl,True
2,duration,XGBoost,1.000,1.000,0.000,outputs\models\best_duration.pkl,False
3,vibration,XGBoost,0.516,0.510,0.484,outputs\models\best_vibration.pkl,True
4,contact_stress,RandomForest,0.588,0.577,0.412,outputs\models\best_contact_stress.pkl,True
5,posture,RandomForest,0.891,0.765,0.109,outputs\models\best_posture.pkl,False
